In [2]:
import torch
import torch.nn as nn

In [95]:
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction_rate=2):
        super(ChannelAttention, self).__init__()

        self.squeeze = nn.ModuleList([
            nn.AdaptiveAvgPool2d(1),
            nn.AdaptiveMaxPool2d(1)
        ])

        self.excitation = nn.Sequential(
            nn.Conv2d(in_channels=channels,
                      out_channels=channels // reduction_rate,
                      kernel_size=1,
                      bias=False),
            nn.SiLU(),
            nn.Conv2d(in_channels=channels // reduction_rate,
                      out_channels=channels,
                      kernel_size=1,
                      bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # perform squeeze with independent Pooling
        avg_feat = self.squeeze[0](x)
        max_feat = self.squeeze[1](x)
        # perform excitation with the same excitation sub-net
        avg_out = self.excitation(avg_feat)
        max_out = self.excitation(max_feat)
        # attention
        attention = self.sigmoid(avg_out + max_out)
        return attention * x
ca = ChannelAttention(4)

In [96]:
image = torch.rand(1, 3, 8, 8)
image = nn.Conv2d(3, 4, 3, bias=False)(image)

In [97]:
image

tensor([[[[ 0.1743,  0.4497,  0.2306,  0.0794,  0.2120,  0.2618],
          [ 0.0098,  0.3278,  0.0760, -0.1437,  0.0036,  0.1107],
          [ 0.4554,  0.4144,  0.5569,  0.2362,  0.1813,  0.5071],
          [ 0.1330,  0.0606,  0.5583,  0.2684,  0.3132,  0.4244],
          [ 0.0226,  0.3602,  0.5880,  0.2602,  0.4356, -0.0477],
          [ 0.0755,  0.1488,  0.2007,  0.3472,  0.0525, -0.0714]],

         [[ 0.1173,  0.0611, -0.0572,  0.1300,  0.2964, -0.0119],
          [-0.0487, -0.0261,  0.2986, -0.0472, -0.3953, -0.2145],
          [ 0.1423, -0.1430, -0.2215,  0.3059,  0.1553,  0.0091],
          [ 0.1110, -0.3527, -0.2324, -0.1796,  0.1107, -0.0077],
          [-0.3372, -0.1640, -0.2578,  0.2496, -0.1605, -0.1226],
          [ 0.1112, -0.1565, -0.0330, -0.1592, -0.0078, -0.2159]],

         [[ 0.2817,  0.1103,  0.3790,  0.4191,  0.1823, -0.0067],
          [ 0.1058,  0.3466,  0.3321,  0.3729,  0.2727,  0.4306],
          [ 0.3669,  0.3703,  0.4289,  0.2791,  0.1798,  0.1409],
      

In [98]:
a = ca(image)

In [99]:
a

tensor([[[[ 0.0909,  0.2346,  0.1203,  0.0414,  0.1106,  0.1366],
          [ 0.0051,  0.1710,  0.0396, -0.0750,  0.0019,  0.0578],
          [ 0.2376,  0.2162,  0.2905,  0.1232,  0.0946,  0.2646],
          [ 0.0694,  0.0316,  0.2913,  0.1400,  0.1634,  0.2214],
          [ 0.0118,  0.1879,  0.3067,  0.1357,  0.2273, -0.0249],
          [ 0.0394,  0.0776,  0.1047,  0.1811,  0.0274, -0.0372]],

         [[ 0.0619,  0.0322, -0.0302,  0.0686,  0.1563, -0.0063],
          [-0.0257, -0.0138,  0.1575, -0.0249, -0.2085, -0.1131],
          [ 0.0751, -0.0754, -0.1168,  0.1614,  0.0819,  0.0048],
          [ 0.0586, -0.1860, -0.1226, -0.0948,  0.0584, -0.0040],
          [-0.1779, -0.0865, -0.1360,  0.1316, -0.0847, -0.0647],
          [ 0.0587, -0.0826, -0.0174, -0.0840, -0.0041, -0.1139]],

         [[ 0.1397,  0.0547,  0.1879,  0.2078,  0.0904, -0.0033],
          [ 0.0525,  0.1718,  0.1647,  0.1849,  0.1352,  0.2135],
          [ 0.1819,  0.1836,  0.2126,  0.1384,  0.0892,  0.0699],
      